In [1]:
# ============================================================
# EXP-1) Tempo/Rate Shift (test-time time-warp) — Trial-level Count Error
#   - 6 activities (MHEALTH): act_id = [6,7,8,10,11,12]
#   - LOSO (train on 9 subjects, test on 1 subject) per activity
#   - Compare: Ours vs B0 (terminal count regression baseline)
#   - SAME windowing, SAME count-only supervision (window y_count), SAME pipeline
#   - Only difference for B0: loss = terminal count regression (per window),
#       and backbone capacity increased (hidden_dim*2, encoder/decoder layer +1)
#   - NO visualization
#   - Output: raw_rows.csv + summary_by_activity_mult.csv + summary_overall_mult.csv
# ============================================================

import os
import glob
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# 1) Strict Seeding
# ------------------------------------------------------------
def set_strict_seed(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# 2) Data Loading (MHEALTH)
# ------------------------------------------------------------
def load_mhealth_dataset(data_dir, target_activities_map, column_names):
    """
    Returns:
      full_dataset[subj_key][activity_name] = pd.DataFrame(features only, no activity_id)
    """
    full_dataset = {}
    file_list = sorted(glob.glob(os.path.join(data_dir, "mHealth_subject*.log")))

    if not file_list:
        print(f"[Warning] No mHealth logs found in {data_dir}")
        return {}

    for file_path in file_list:
        file_name = os.path.basename(file_path)
        subj_part = file_name.split('.')[0]
        try:
            subj_id_num = int(''.join(filter(str.isdigit, subj_part)))
            subj_key = f"subject{subj_id_num}"
        except:
            subj_key = subj_part

        try:
            df = pd.read_csv(file_path, sep="\t", header=None)
            df = df.iloc[:, :len(column_names)]
            df.columns = column_names

            subj_data = {}
            for label_code, activity_name in target_activities_map.items():
                activity_df = df[df['activity_id'] == label_code].copy()
                if not activity_df.empty:
                    subj_data[activity_name] = activity_df.drop(columns=['activity_id'])

            full_dataset[subj_key] = subj_data
        except Exception as e:
            print(f"[Error] loading {file_name}: {e}")
            continue

    return full_dataset


def prepare_trial_list(label_config, full_data, target_map, feature_map):
    """
    label_config: list of (subj, act_id, gt_count)
    Creates ONE trial per (subj, act_id) with trial-wise z-score normalization.
    """
    trial_list = []
    for subj, act_id, gt_count in label_config:
        act_name = target_map.get(act_id)
        feats = feature_map.get(act_id)

        if subj in full_data and act_name in full_data[subj]:
            raw_df = full_data[subj][act_name][feats]
            raw_np = raw_df.values.astype(np.float32)

            # Trial-wise z-score
            mean = raw_np.mean(axis=0)
            std = raw_np.std(axis=0) + 1e-6
            norm_np = (raw_np - mean) / std

            trial_list.append({
                "data": norm_np,              # (T,C)
                "count": float(gt_count),      # trial total count
                "meta": f"{subj}_{act_name}",
                "subj": subj,
                "act_id": int(act_id),
            })
        else:
            # missing or empty
            pass

    return trial_list


# ------------------------------------------------------------
# 2.5) Windowing
# ------------------------------------------------------------
def trial_list_to_windows(trial_list, fs, win_sec=8.0, stride_sec=4.0, drop_last=True):
    """
    TRAIN-only: trial -> sliding windows
      rate_trial = count_total / total_duration
      count_window = rate_trial * window_duration
    """
    win_len = int(round(win_sec * fs))
    stride = int(round(stride_sec * fs))
    assert win_len > 0 and stride > 0

    windows = []
    for item in trial_list:
        x = item["data"]  # (T,C)
        T = x.shape[0]
        total_count = float(item["count"])
        meta = item["meta"]

        total_dur = max(T / float(fs), 1e-6)
        rate_trial = total_count / total_dur  # reps/s

        if T < win_len:
            win_dur = T / float(fs)
            windows.append({
                "data": x,
                "count": rate_trial * win_dur,  # window count label
                "meta": f"{meta}__win[0:{T}]",
            })
            continue

        last_start = T - win_len
        starts = list(range(0, last_start + 1, stride))

        for st in starts:
            ed = st + win_len
            win_dur = win_len / float(fs)
            windows.append({
                "data": x[st:ed],
                "count": rate_trial * win_dur,
                "meta": f"{meta}__win[{st}:{ed}]",
            })

        if not drop_last:
            last_st = starts[-1] + stride
            if last_st < T:
                ed = T
                win_dur = (ed - last_st) / float(fs)
                windows.append({
                    "data": x[last_st:ed],
                    "count": rate_trial * win_dur,
                    "meta": f"{meta}__win[{last_st}:{ed}]",
                })

    return windows


def predict_count_by_windowing(model, x_np, fs, win_sec, stride_sec, device, tau=1.0, batch_size=64):
    """
    TEST-only: trial -> windows inference -> mean rate -> total count
    x_np: (T,C) numpy (normalized)
    """
    win_len = int(round(win_sec * fs))
    stride = int(round(stride_sec * fs))
    T = x_np.shape[0]
    total_dur = T / float(fs)

    model.eval()

    # short trial -> 1 forward
    if T <= win_len:
        x_tensor = torch.tensor(x_np, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(device)  # (1,C,T)
        with torch.no_grad():
            rate_hat, _, _, _ = model(x_tensor, mask=None, tau=tau)
        pred_count = float(rate_hat.item() * total_dur)
        return pred_count

    starts = list(range(0, T - win_len + 1, stride))
    windows = np.stack([x_np[st:st + win_len] for st in starts], axis=0)  # (N, win_len, C)
    xw = torch.tensor(windows, dtype=torch.float32).permute(0, 2, 1).to(device)  # (N,C,win_len)

    rates = []
    with torch.no_grad():
        for i in range(0, xw.shape[0], batch_size):
            xb = xw[i:i + batch_size]
            r_hat, _, _, _ = model(xb, mask=None, tau=tau)
            rates.append(r_hat.detach().cpu().numpy())

    rates = np.concatenate(rates, axis=0)  # (N,)
    rate_mean = float(rates.mean())
    pred_count = rate_mean * total_dur
    return float(pred_count)


# ------------------------------------------------------------
# 2.6) Time-warp (TEST-only)
# ------------------------------------------------------------
def time_warp_linear(x_np: np.ndarray, dur_mult: float) -> np.ndarray:
    """
    x_np: (T,C)
    dur_mult > 1 => slower (T increases)
    dur_mult < 1 => faster (T decreases)
    """
    assert x_np.ndim == 2, f"x_np must be (T,C), got {x_np.shape}"
    T, C = x_np.shape
    new_T = int(round(T * float(dur_mult)))
    new_T = max(new_T, 2)

    t_old = np.linspace(0.0, 1.0, T, dtype=np.float32)
    t_new = np.linspace(0.0, 1.0, new_T, dtype=np.float32)

    out = np.zeros((new_T, C), dtype=np.float32)
    for c in range(C):
        out[:, c] = np.interp(t_new, t_old, x_np[:, c]).astype(np.float32)
    return out


# ------------------------------------------------------------
# 2.8) Dataset / Collate
# ------------------------------------------------------------
class TrialDataset(Dataset):
    def __init__(self, windows_list):
        self.items = windows_list

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        data = torch.tensor(item["data"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        count = torch.tensor(item["count"], dtype=torch.float32)               # window count
        return data, count, item["meta"]


def collate_variable_length(batch):
    max_len = max([x[0].shape[1] for x in batch])
    C = batch[0][0].shape[0]

    padded_data, masks, counts, metas, lengths = [], [], [], [], []
    for data, count, meta in batch:
        T = data.shape[1]
        lengths.append(T)

        pad_size = max_len - T
        if pad_size > 0:
            pad = torch.zeros(C, pad_size)
            d_padded = torch.cat([data, pad], dim=1)
            mask = torch.cat([torch.ones(T), torch.zeros(pad_size)], dim=0)
        else:
            d_padded = data
            mask = torch.ones(T)

        padded_data.append(d_padded)
        masks.append(mask)
        counts.append(count)
        metas.append(meta)

    return {
        "data": torch.stack(padded_data),                       # (B,C,Tmax)
        "mask": torch.stack(masks),                             # (B,Tmax)
        "count": torch.stack(counts),                           # (B,)
        "length": torch.tensor(lengths, dtype=torch.float32),   # (B,)
        "meta": metas
    }


# ------------------------------------------------------------
# 3) Models
# ------------------------------------------------------------
class ManifoldEncoder(nn.Module):
    def __init__(self, input_ch, hidden_dim=128, latent_dim=16, n_blocks=2):
        super().__init__()
        layers = []
        in_ch = input_ch
        for _ in range(n_blocks):
            layers += [
                nn.Conv1d(in_ch, hidden_dim, 5, padding=2),
                nn.ReLU(),
            ]
            in_ch = hidden_dim
        layers += [nn.Conv1d(hidden_dim, latent_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        z = self.net(x)            # (B,D,T)
        z = z.transpose(1, 2)      # (B,T,D)
        return z


class ManifoldDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, out_ch, n_blocks=2):
        super().__init__()
        layers = []
        in_ch = latent_dim
        for _ in range(n_blocks):
            layers += [
                nn.Conv1d(in_ch, hidden_dim, 5, padding=2),
                nn.ReLU(),
            ]
            in_ch = hidden_dim
        layers += [nn.Conv1d(hidden_dim, out_ch, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        zt = z.transpose(1, 2)     # (B,D,T)
        x_hat = self.net(zt)       # (B,C,T)
        return x_hat


class MultiRateHead(nn.Module):
    def __init__(self, latent_dim=16, hidden=64, K_max=6):
        super().__init__()
        self.K_max = K_max
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1 + K_max)  # [amp | phase_logits...]
        )

    def forward(self, z, tau=1.0):
        out = self.net(z)                     # (B,T,1+K)
        amp = F.softplus(out[..., 0])         # (B,T) >=0
        phase_logits = out[..., 1:]           # (B,T,K)
        phase = F.softmax(phase_logits / tau, dim=-1)
        return amp, phase, phase_logits


class KAutoCountModel(nn.Module):
    """
    Ours model (also reused for B0 with different capacity).
    """
    def __init__(self, input_ch, hidden_dim=128, latent_dim=16, K_max=6, enc_blocks=2, dec_blocks=2):
        super().__init__()
        self.encoder = ManifoldEncoder(input_ch, hidden_dim, latent_dim, n_blocks=enc_blocks)
        self.decoder = ManifoldDecoder(latent_dim, hidden_dim, input_ch, n_blocks=dec_blocks)
        self.rate_head = MultiRateHead(latent_dim, hidden=hidden_dim, K_max=K_max)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

        with torch.no_grad():
            b = self.rate_head.net[-1].bias
            b.zero_()
            b[0].fill_(-2.0)

    @staticmethod
    def _masked_mean_time(x, mask=None, eps=1e-6):
        if mask is None:
            return x.mean(dim=1)
        if x.dim() == 2:
            m = mask.to(dtype=x.dtype, device=x.device)
            return (x * m).sum(dim=1) / (m.sum(dim=1) + eps)
        elif x.dim() == 3:
            m = mask.to(dtype=x.dtype, device=x.device).unsqueeze(-1)
            return (x * m).sum(dim=1) / (m.sum(dim=1) + eps)
        else:
            raise ValueError(f"Unsupported dim for masked mean: {x.dim()}")

    def forward(self, x, mask=None, tau=1.0):
        z = self.encoder(x)              # (B,T,D)
        x_hat = self.decoder(z)          # (B,C,T)

        amp_t, phase_p, phase_logits = self.rate_head(z, tau=tau)
        micro_rate_t = amp_t             # (B,T)

        p_bar = self._masked_mean_time(phase_p, mask)           # (B,K)
        k_hat = 1.0 / (p_bar.pow(2).sum(dim=1) + 1e-6)          # (B,)

        rep_rate_t = micro_rate_t / (k_hat.unsqueeze(1) + 1e-6) # (B,T)
        if mask is not None:
            rep_rate_t = rep_rate_t * mask

        if mask is None:
            avg_rep_rate = rep_rate_t.mean(dim=1)
        else:
            avg_rep_rate = (rep_rate_t * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-6)

        aux = {
            "phase_p": phase_p,          # (B,T,K)
            "rep_rate_t": rep_rate_t,    # (B,T)
            "k_hat": k_hat,              # (B,)
        }
        return avg_rep_rate, z, x_hat, aux


# ------------------------------------------------------------
# 4) Loss utils (Ours)
# ------------------------------------------------------------
def masked_recon_mse(x_hat, x, mask, eps=1e-6):
    mask = mask.to(dtype=x.dtype, device=x.device)
    mask_bc = mask.unsqueeze(1)              # (B,1,T)
    se = (x_hat - x) ** 2                    # (B,C,T)
    se = se * mask_bc
    denom = (mask.sum() * x.shape[1]) + eps
    return se.sum() / denom


def temporal_smoothness(v, mask=None, eps=1e-6):
    dv = torch.abs(v[:, 1:] - v[:, :-1])
    if mask is None:
        return dv.mean()
    m = mask[:, 1:] * mask[:, :-1]
    m = m.to(dtype=dv.dtype, device=dv.device)
    return (dv * m).sum() / (m.sum() + eps)


def phase_entropy_loss(phase_p, mask=None, eps=1e-8):
    ent = -(phase_p * (phase_p + eps).log()).sum(dim=-1)  # (B,T)
    if mask is None:
        return ent.mean()
    ent = ent * mask
    return ent.sum() / (mask.sum() + eps)


def effK_usage_loss(phase_p, mask=None, eps=1e-6):
    if mask is None:
        p_bar = phase_p.mean(dim=1)
    else:
        m = mask.to(dtype=phase_p.dtype, device=phase_p.device).unsqueeze(-1)
        p_bar = (phase_p * m).sum(dim=1) / (m.sum(dim=1) + eps)
    effK = 1.0 / (p_bar.pow(2).sum(dim=1) + eps)
    return effK.mean()


# ------------------------------------------------------------
# 5) Train loops
# ------------------------------------------------------------
def train_one_epoch_ours(model, loader, optimizer, config, device):
    model.train()
    fs = config["fs"]
    tau = config.get("tau", 1.0)

    lam_recon = config.get("lambda_recon", 1.0)
    lam_smooth = config.get("lambda_smooth", 0.05)
    lam_phase_ent = config.get("lambda_phase_ent", 0.01)
    lam_effk = config.get("lambda_effk", 0.005)

    for batch in loader:
        x = batch["data"].to(device)
        mask = batch["mask"].to(device)
        y_count = batch["count"].to(device)
        length = batch["length"].to(device)

        duration = torch.clamp(length / fs, min=1e-6)
        y_rate = y_count / duration

        optimizer.zero_grad()
        rate_hat, _, x_hat, aux = model(x, mask, tau=tau)

        loss_rate = F.mse_loss(rate_hat, y_rate)
        loss_recon = masked_recon_mse(x_hat, x, mask)
        loss_smooth = temporal_smoothness(aux["rep_rate_t"], mask)
        loss_phase_ent = phase_entropy_loss(aux["phase_p"], mask)
        loss_effk = effK_usage_loss(aux["phase_p"], mask)

        loss = (loss_rate
                + lam_recon * loss_recon
                + lam_smooth * loss_smooth
                + lam_phase_ent * loss_phase_ent
                + lam_effk * loss_effk)

        loss.backward()
        optimizer.step()


def train_one_epoch_b0_terminal(model, loader, optimizer, config, device):
    """
    B0: SAME forward (rate prediction), but supervision is terminal window count regression.
      count_hat_win = rate_hat * duration_win
      loss = MSE(count_hat_win, y_count_win)
    """
    model.train()
    fs = config["fs"]
    tau = config.get("tau", 1.0)

    for batch in loader:
        x = batch["data"].to(device)
        mask = batch["mask"].to(device)
        y_count = batch["count"].to(device)
        length = batch["length"].to(device)

        duration = torch.clamp(length / fs, min=1e-6)

        optimizer.zero_grad()
        rate_hat, _, _, _ = model(x, mask, tau=tau)

        count_hat = rate_hat * duration
        loss = F.mse_loss(count_hat, y_count)

        loss.backward()
        optimizer.step()


# ------------------------------------------------------------
# 6) Metrics
# ------------------------------------------------------------
def mae_mape(pred, gt, eps=1e-6):
    pred = float(pred)
    gt = float(gt)
    mae = abs(pred - gt)
    mape = mae / (abs(gt) + eps) * 100.0
    return mae, mape


# ------------------------------------------------------------
# 7) Main
# ------------------------------------------------------------
def main():
    # -----------------------------
    # CONFIG
    # -----------------------------
    CONFIG = {
        "seed": 42,
        "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",

        "COLUMN_NAMES": [
            'acc_chest_x', 'acc_chest_y', 'acc_chest_z',
            'ecg_1', 'ecg_2',
            'acc_ankle_x', 'acc_ankle_y', 'acc_ankle_z',
            'gyro_ankle_x', 'gyro_ankle_y', 'gyro_ankle_z',
            'mag_ankle_x', 'mag_ankle_y', 'mag_ankle_z',
            'acc_arm_x', 'acc_arm_y', 'acc_arm_z',
            'gyro_arm_x', 'gyro_arm_y', 'gyro_arm_z',
            'mag_arm_x', 'mag_arm_y', 'mag_arm_z',
            'activity_id'
        ],

        # Training Params
        "epochs": 100,
        "lr": 5e-4,
        "batch_size": 64,
        "fs": 50,

        # Windowing Params
        "win_sec": 8.0,
        "stride_sec": 4.0,
        "drop_last": True,

        # Ours model capacity
        "hidden_dim": 128,
        "latent_dim": 16,
        "K_max": 6,
        "enc_blocks": 2,
        "dec_blocks": 2,

        # Ours loss weights
        "lambda_recon": 1.0,
        "lambda_smooth": 0.05,
        "lambda_phase_ent": 0.01,
        "lambda_effk": 0.0075,

        "tau": 1.0,
    }

    # B0 capacity rule: hidden_dim*2, layer +1 (encoder/decoder blocks +1)
    B0_CAPACITY = {
        "hidden_dim_mul": 2,
        "extra_blocks": 1,  # enc_blocks+1, dec_blocks+1
    }

    # Test-time tempo warp multipliers
    MULTS = [0.6, 0.7, 0.75, 0.8, 0.9, 1.0, 1.1, 1.25, 1.4, 1.6]

    # Output
    OUT_DIR = "exp1_rate_shift_outputs"
    os.makedirs(OUT_DIR, exist_ok=True)

    # -----------------------------
    # Activity Specs (GT)
    # -----------------------------
    ACTIVITY_SPECS = [
        {"act_id": 6,  "act_name": "Waist bends forward",       "labels": [
            ("subject1", 6, 21), ("subject2", 6, 19), ("subject3", 6, 21), ("subject4", 6, 20), ("subject5", 6, 20),
            ("subject6", 6, 20), ("subject7", 6, 20), ("subject8", 6, 21), ("subject9", 6, 21), ("subject10", 6, 20),
        ]},
        {"act_id": 7,  "act_name": "Frontal elevation of arms", "labels": [
            ("subject1", 7, 20), ("subject2", 7, 20), ("subject3", 7, 20), ("subject4", 7, 20), ("subject5", 7, 20),
            ("subject6", 7, 20), ("subject7", 7, 20), ("subject8", 7, 19), ("subject9", 7, 19), ("subject10", 7, 20),
        ]},
        {"act_id": 8,  "act_name": "Knees bending",             "labels": [
            ("subject1", 8, 20), ("subject2", 8, 21), ("subject3", 8, 21), ("subject4", 8, 19), ("subject5", 8, 20),
            ("subject6", 8, 20), ("subject7", 8, 21), ("subject8", 8, 21), ("subject9", 8, 21), ("subject10", 8, 21),
        ]},
        {"act_id": 12, "act_name": "Jump front & back",         "labels": [
            ("subject1", 12, 20), ("subject2", 12, 22), ("subject3", 12, 21), ("subject4", 12, 21), ("subject5", 12, 20),
            ("subject6", 12, 21), ("subject7", 12, 19), ("subject8", 12, 20), ("subject9", 12, 20), ("subject10", 12, 20),
        ]},
        {"act_id": 10, "act_name": "Jogging",                   "labels": [
            ("subject1", 10, 157), ("subject2", 10, 161), ("subject3", 10, 154), ("subject4", 10, 154), ("subject5", 10, 160),
            ("subject6", 10, 156), ("subject7", 10, 153), ("subject8", 10, 160), ("subject9", 10, 166), ("subject10", 10, 156),
        ]},
        {"act_id": 11, "act_name": "Running",                   "labels": [
            ("subject1", 11, 165), ("subject2", 11, 158), ("subject3", 11, 174), ("subject4", 11, 163), ("subject5", 11, 157),
            ("subject6", 11, 172), ("subject7", 11, 149), ("subject8", 11, 166), ("subject9", 11, 174), ("subject10", 11, 172),
        ]},
    ]

    # Feature map (same set used before: acc/gyro for chest/ankle/arm)
    COMMON_FEATURES = [
        'acc_chest_x', 'acc_chest_y', 'acc_chest_z',
        'acc_ankle_x', 'acc_ankle_y', 'acc_ankle_z',
        'gyro_ankle_x', 'gyro_ankle_y', 'gyro_ankle_z',
        'acc_arm_x', 'acc_arm_y', 'acc_arm_z',
        'gyro_arm_x', 'gyro_arm_y', 'gyro_arm_z'
    ]

    TARGET_ACTIVITIES_MAP = {spec["act_id"]: spec["act_name"] for spec in ACTIVITY_SPECS}
    ACT_FEATURE_MAP = {spec["act_id"]: COMMON_FEATURES for spec in ACTIVITY_SPECS}

    # Subjects list
    subjects = [f"subject{i}" for i in range(1, 11)]

    # -----------------------------
    # Init
    # -----------------------------
    set_strict_seed(CONFIG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")

    full_data = load_mhealth_dataset(CONFIG["data_dir"], TARGET_ACTIVITIES_MAP, CONFIG["COLUMN_NAMES"])
    if not full_data:
        print("[Abort] Dataset load failed.")
        return

    # -----------------------------
    # Run Experiment
    # -----------------------------
    all_rows = []

    for spec in ACTIVITY_SPECS:
        act_id = int(spec["act_id"])
        act_name = spec["act_name"]
        labels_all = spec["labels"]

        print("\n" + "=" * 92)
        print(f"[Activity] act_id={act_id} | {act_name}")
        print("=" * 92)

        for fold_idx, test_subj in enumerate(subjects):
            set_strict_seed(CONFIG["seed"])

            train_subjects = [s for s in subjects if s != test_subj]
            test_subjects = [test_subj]

            # labels split
            train_labels = [(subj, act_id, gt) for (subj, a, gt) in labels_all if subj in train_subjects]
            test_labels = [(subj, act_id, gt) for (subj, a, gt) in labels_all if subj in test_subjects]

            train_trials = prepare_trial_list(train_labels, full_data, TARGET_ACTIVITIES_MAP, ACT_FEATURE_MAP)
            test_trials = prepare_trial_list(test_labels, full_data, TARGET_ACTIVITIES_MAP, ACT_FEATURE_MAP)

            if len(train_trials) == 0 or len(test_trials) == 0:
                print(f"[Skip] {act_name} fold {fold_idx+1} test={test_subj} (missing trials).")
                continue

            # train windows (same for both models)
            train_windows = trial_list_to_windows(
                train_trials,
                fs=CONFIG["fs"],
                win_sec=CONFIG["win_sec"],
                stride_sec=CONFIG["stride_sec"],
                drop_last=CONFIG["drop_last"]
            )

            g = torch.Generator()
            g.manual_seed(CONFIG["seed"])

            train_loader = DataLoader(
                TrialDataset(train_windows),
                batch_size=CONFIG["batch_size"],
                shuffle=True,
                collate_fn=collate_variable_length,
                generator=g,
                num_workers=0
            )

            input_ch = train_windows[0]["data"].shape[1]

            # -----------------------------
            # (1) Train OURS (once)
            # -----------------------------
            ours = KAutoCountModel(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"],
                latent_dim=CONFIG["latent_dim"],
                K_max=CONFIG["K_max"],
                enc_blocks=CONFIG["enc_blocks"],
                dec_blocks=CONFIG["dec_blocks"]
            ).to(device)

            opt_ours = torch.optim.Adam(ours.parameters(), lr=CONFIG["lr"])
            sch_ours = torch.optim.lr_scheduler.StepLR(opt_ours, step_size=30, gamma=0.5)

            for _ in range(CONFIG["epochs"]):
                train_one_epoch_ours(ours, train_loader, opt_ours, CONFIG, device)
                sch_ours.step()

            # -----------------------------
            # (2) Train B0 (once)
            # -----------------------------
            b0_hidden = int(CONFIG["hidden_dim"] * B0_CAPACITY["hidden_dim_mul"])
            b0_enc_blocks = int(CONFIG["enc_blocks"] + B0_CAPACITY["extra_blocks"])
            b0_dec_blocks = int(CONFIG["dec_blocks"] + B0_CAPACITY["extra_blocks"])

            b0 = KAutoCountModel(
                input_ch=input_ch,
                hidden_dim=b0_hidden,
                latent_dim=CONFIG["latent_dim"],
                K_max=CONFIG["K_max"],
                enc_blocks=b0_enc_blocks,
                dec_blocks=b0_dec_blocks
            ).to(device)

            opt_b0 = torch.optim.Adam(b0.parameters(), lr=CONFIG["lr"])
            sch_b0 = torch.optim.lr_scheduler.StepLR(opt_b0, step_size=30, gamma=0.5)

            for _ in range(CONFIG["epochs"]):
                train_one_epoch_b0_terminal(b0, train_loader, opt_b0, CONFIG, device)
                sch_b0.step()

            # -----------------------------
            # Test trial (single trial for this subject/activity)
            # -----------------------------
            test_item = test_trials[0]
            x_raw = test_item["data"]           # (T,C) normalized
            gt_count = float(test_item["count"])

            # Sweep mults (TEST-ONLY time-warp)
            for mult in MULTS:
                x_warp = time_warp_linear(x_raw, float(mult))

                pred_ours = predict_count_by_windowing(
                    ours, x_warp,
                    fs=CONFIG["fs"],
                    win_sec=CONFIG["win_sec"],
                    stride_sec=CONFIG["stride_sec"],
                    device=device,
                    tau=CONFIG.get("tau", 1.0),
                    batch_size=CONFIG["batch_size"]
                )

                pred_b0 = predict_count_by_windowing(
                    b0, x_warp,
                    fs=CONFIG["fs"],
                    win_sec=CONFIG["win_sec"],
                    stride_sec=CONFIG["stride_sec"],
                    device=device,
                    tau=CONFIG.get("tau", 1.0),
                    batch_size=CONFIG["batch_size"]
                )

                mae_o, mape_o = mae_mape(pred_ours, gt_count)
                mae_b, mape_b = mae_mape(pred_b0, gt_count)

                all_rows.append({
                    "act_id": act_id,
                    "act_name": act_name,
                    "test_subj": test_subj,
                    "mult": float(mult),
                    "gt_count": gt_count,
                    "pred_ours": float(pred_ours),
                    "pred_b0": float(pred_b0),
                    "mae_ours": float(mae_o),
                    "mape_ours": float(mape_o),
                    "mae_b0": float(mae_b),
                    "mape_b0": float(mape_b),
                })

            print(f"[Done] {act_name} | fold {fold_idx+1:2d}/10 | test={test_subj}")

    if len(all_rows) == 0:
        print("[Abort] No results generated.")
        return

    # -----------------------------
    # Save raw rows
    # -----------------------------
    df_raw = pd.DataFrame(all_rows)
    raw_path = os.path.join(OUT_DIR, "raw_rows.csv")
    df_raw.to_csv(raw_path, index=False)
    print(f"\n[Saved] raw rows -> {raw_path}")

    # -----------------------------
    # Summary by (activity, mult)
    # -----------------------------
    df_sum_act = (
        df_raw.groupby(["act_id", "act_name", "mult"])
        .agg(
            n=("test_subj", "count"),
            ours_MAE_mean=("mae_ours", "mean"),
            ours_MAE_std=("mae_ours", "std"),
            ours_MAPE_mean=("mape_ours", "mean"),
            ours_MAPE_std=("mape_ours", "std"),
            b0_MAE_mean=("mae_b0", "mean"),
            b0_MAE_std=("mae_b0", "std"),
            b0_MAPE_mean=("mape_b0", "mean"),
            b0_MAPE_std=("mape_b0", "std"),
        )
        .reset_index()
        .sort_values(["act_id", "mult"])
        .reset_index(drop=True)
    )

    sum_act_path = os.path.join(OUT_DIR, "summary_by_activity_mult.csv")
    df_sum_act.to_csv(sum_act_path, index=False)
    print(f"[Saved] summary (activity×mult) -> {sum_act_path}")

    # -----------------------------
    # Overall summary by mult (across all activities + subjects)
    # -----------------------------
    df_sum_all = (
        df_raw.groupby(["mult"])
        .agg(
            n=("test_subj", "count"),
            ours_MAE_mean=("mae_ours", "mean"),
            ours_MAE_std=("mae_ours", "std"),
            ours_MAPE_mean=("mape_ours", "mean"),
            ours_MAPE_std=("mape_ours", "std"),
            b0_MAE_mean=("mae_b0", "mean"),
            b0_MAE_std=("mae_b0", "std"),
            b0_MAPE_mean=("mape_b0", "mean"),
            b0_MAPE_std=("mape_b0", "std"),
        )
        .reset_index()
        .sort_values(["mult"])
        .reset_index(drop=True)
    )

    sum_all_path = os.path.join(OUT_DIR, "summary_overall_mult.csv")
    df_sum_all.to_csv(sum_all_path, index=False)
    print(f"[Saved] summary (overall×mult) -> {sum_all_path}")

    # Optional: print a compact view
    with pd.option_context("display.max_rows", 200, "display.max_columns", 50, "display.width", 160):
        print("\n" + "-" * 92)
        print("[Overall summary by mult]")
        print("-" * 92)
        print(df_sum_all.to_string(index=False))


if __name__ == "__main__":
    main()


[Device] cuda

[Activity] act_id=6 | Waist bends forward
[Done] Waist bends forward | fold  1/10 | test=subject1
[Done] Waist bends forward | fold  2/10 | test=subject2
[Done] Waist bends forward | fold  3/10 | test=subject3
[Done] Waist bends forward | fold  4/10 | test=subject4
[Done] Waist bends forward | fold  5/10 | test=subject5
[Done] Waist bends forward | fold  6/10 | test=subject6
[Done] Waist bends forward | fold  7/10 | test=subject7
[Done] Waist bends forward | fold  8/10 | test=subject8
[Done] Waist bends forward | fold  9/10 | test=subject9
[Done] Waist bends forward | fold 10/10 | test=subject10

[Activity] act_id=7 | Frontal elevation of arms
[Done] Frontal elevation of arms | fold  1/10 | test=subject1
[Done] Frontal elevation of arms | fold  2/10 | test=subject2
[Done] Frontal elevation of arms | fold  3/10 | test=subject3
[Done] Frontal elevation of arms | fold  4/10 | test=subject4
[Done] Frontal elevation of arms | fold  5/10 | test=subject5
[Done] Frontal elevatio